# M5 Baseline LightGBM Pipeline — Internal Features E5~E9 (Auto-run)

This notebook tests internal feature groups E5~E9 using an auto-run loop.

- **E5**: Price Enhancement (price_vs_mean, price_vs_dept_avg)
- **E6**: Promotion Features (is_price_drop, is_price_min_7d)
- **E7**: Event & SNAP (snap_flag, is_event)
- **E8**: Hierarchy Features (dept_avg_demand, store_avg_demand)
- **E9**: Combined (E5+E6+E7+E8)

In [1]:
# =========================
# 1) Colab setup
# =========================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================
# 2) Imports
# =========================
from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 200)

In [3]:
# =========================
# 3) Configuration
# =========================
DATA_DIR = Path("/content/drive/MyDrive/Group Project - Predictive/data/cleaned_data")
CLEAN_DATA_PATH = DATA_DIR / "long_df_clean.parquet"

OUTPUT_DIR = Path("/content/drive/MyDrive/Group Project - Predictive/data/FE_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE = True
N_SERIES = 2000
RANDOM_STATE = 1234
VALID_DAYS = 28

PIPELINE_NAME = "baseline_lgb_clean_minimal_features"

print("CLEAN_DATA_PATH:", CLEAN_DATA_PATH)
print("OUTPUT_DIR      :", OUTPUT_DIR)

CLEAN_DATA_PATH: /content/drive/MyDrive/Group Project - Predictive/data/cleaned_data/long_df_clean.parquet
OUTPUT_DIR      : /content/drive/MyDrive/Group Project - Predictive/data/FE_output


In [4]:
# =========================
# 4) Load cleaned dataset
# =========================
import pyarrow.parquet as pq

needed_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "date", "demand", "sell_price",
    "snap_CA", "snap_TX", "snap_WI",
    "event_name_1"
]
available_cols = pq.read_schema(CLEAN_DATA_PATH).names
needed_cols = [c for c in needed_cols if c in available_cols]

df = pd.read_parquet(CLEAN_DATA_PATH, columns=needed_cols)
print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())

Loaded shape: (16098720, 13)
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'date', 'demand', 'sell_price', 'snap_CA', 'snap_TX', 'snap_WI', 'event_name_1']


In [5]:
# =========================
# 5) Sampling
# =========================
df["date"] = pd.to_datetime(df["date"])

if FAST_MODE:
    sampled_ids = (
        df["id"]
        .drop_duplicates()
        .sample(n=min(N_SERIES, df["id"].nunique()), random_state=RANDOM_STATE)
    )
    df = df[df["id"].isin(sampled_ids)].copy()

gc.collect()
df = df.sort_values(["id", "date"]).reset_index(drop=True)

print("Working shape:", df.shape)
print("Unique series:", df["id"].nunique())
print("Date range   :", df["date"].min().date(), "to", df["date"].max().date())

Working shape: (1056000, 13)
Unique series: 2000
Date range   : 2014-12-12 to 2016-05-22


In [6]:
# =========================
# 6) Baseline feature engineering
# =========================
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# Calendar
df["dow"]   = df["date"].dt.dayofweek.astype("int8")
df["month"] = df["date"].dt.month.astype("int8")

# Price
price_grp = df.groupby("id", sort=False)["sell_price"]
df["price_lag_1"] = price_grp.shift(1)
df["price_change"] = ((df["sell_price"] - df["price_lag_1"]) / df["price_lag_1"]).replace([np.inf, -np.inf], np.nan)
df["price_change"] = df["price_change"].fillna(0).astype("float32")
df["is_promo"] = (df["sell_price"] < df["price_lag_1"].fillna(df["sell_price"])).astype("int8")

# Demand lags & rolling (shift-first to avoid leakage)
demand_grp = df.groupby("id", sort=False)["demand"]
df["lag_7"]  = demand_grp.shift(7)
df["lag_28"] = demand_grp.shift(28)
df["rmean_7"]  = df.groupby("id", sort=False)["lag_7"].transform(lambda x: x.rolling(7).mean())
df["rmean_28"] = df.groupby("id", sort=False)["lag_28"].transform(lambda x: x.rolling(28).mean())

for col in ["sell_price", "price_lag_1", "lag_7", "lag_28", "rmean_7", "rmean_28"]:
    df[col] = pd.to_numeric(df[col], downcast="float")

BASELINE_FEATURE_COLS = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "dow", "month",
    "sell_price", "price_change", "is_promo",
    "lag_7", "lag_28", "rmean_7", "rmean_28"
]

print("Baseline feature engineering done. df shape:", df.shape)

Baseline feature engineering done. df shape: (1056000, 22)


In [7]:
# =========================
# 7) Build all experimental features
# =========================

# E5: Price Enhancement
df["price_vs_mean"] = df["sell_price"] / df.groupby("id")["sell_price"].transform("mean")
dept_daily_avg = df.groupby(["store_id", "dept_id", "date"])["sell_price"].transform("mean")
df["price_vs_dept_avg"] = df["sell_price"] / dept_daily_avg
df[["price_vs_mean", "price_vs_dept_avg"]] = df[["price_vs_mean", "price_vs_dept_avg"]].astype("float32")
print("E5 done")

# E6: Promotion Features
df["is_price_drop"] = (df["sell_price"] < df["price_lag_1"]).astype("int8")
df["price_min_7d"] = df.groupby("id", sort=False)["sell_price"].transform(lambda x: x.shift(1).rolling(7).min())
df["is_price_min_7d"] = (df["sell_price"] == df["price_min_7d"]).astype("int8")
print("E6 done")

# E7: Event & SNAP
df["snap_flag"] = 0
if "snap_CA" in df.columns:
    df.loc[df["state_id"] == "CA", "snap_flag"] = df["snap_CA"]
    df.loc[df["state_id"] == "TX", "snap_flag"] = df["snap_TX"]
    df.loc[df["state_id"] == "WI", "snap_flag"] = df["snap_WI"]
df["snap_flag"] = df["snap_flag"].astype("int8")
if "event_name_1" in df.columns:
    df["event_name_1"] = df["event_name_1"].fillna("NoEvent")
    df["is_event"] = (df["event_name_1"] != "NoEvent").astype("int8")
else:
    df["is_event"] = 0
print("E7 done")

# E8: Hierarchy Features (shift+rolling to avoid leakage)
df["dept_avg_demand"] = df.groupby(["store_id", "dept_id", "date"])["demand"].transform("mean")
df["dept_avg_demand"] = df.groupby("id")["dept_avg_demand"].transform(lambda x: x.shift(28).rolling(28).mean()).astype("float32")

df["store_avg_demand"] = df.groupby(["store_id", "date"])["demand"].transform("mean")
df["store_avg_demand"] = df.groupby("id")["store_avg_demand"].transform(lambda x: x.shift(28).rolling(28).mean()).astype("float32")
print("E8 done")

gc.collect()
print("\nAll experimental features built. df shape:", df.shape)

E5 done
E6 done
E7 done
E8 done

All experimental features built. df shape: (1056000, 31)


In [8]:
# =========================
# 8) LightGBM configuration
# =========================
NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVAL = 100

LGB_PARAMS = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 0.1,
    "seed": RANDOM_STATE,
    "verbosity": -1
}

print(json.dumps(LGB_PARAMS, indent=2))

{
  "objective": "tweedie",
  "tweedie_variance_power": 1.5,
  "metric": "rmse",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 127,
  "min_data_in_leaf": 100,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 1,
  "lambda_l1": 0.0,
  "lambda_l2": 0.1,
  "seed": 1234,
  "verbosity": -1
}


In [9]:
# =========================
# 9) Auto-run E5 ~ E9
# =========================

experiment_addons = {
    "E5": ["price_vs_mean", "price_vs_dept_avg"],
    "E6": ["is_price_drop", "is_price_min_7d"],
    "E7": ["snap_flag", "is_event"],
    "E8": ["dept_avg_demand", "store_avg_demand"],
    "E9": ["price_vs_mean", "price_vs_dept_avg",
           "is_price_drop", "is_price_min_7d",
           "snap_flag", "is_event",
           "dept_avg_demand", "store_avg_demand"]
}

experiment_required_addons = {
    "E5": [],
    "E6": [],
    "E7": [],
    "E8": ["dept_avg_demand", "store_avg_demand"],
    "E9": ["dept_avg_demand", "store_avg_demand"]
}

experiments_to_run = ["E5", "E6", "E7", "E8", "E9"]
cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
results = []

for exp_name in experiments_to_run:
    print("=" * 70)
    print(f"Running: {exp_name} | Added: {experiment_addons[exp_name]}")

    feature_cols = BASELINE_FEATURE_COLS + experiment_addons[exp_name]
    required_history_cols = ["lag_28", "rmean_28"] + experiment_required_addons[exp_name]

    model_df = df.dropna(subset=required_history_cols).copy()
    for col in cat_cols:
        model_df[col] = model_df[col].astype("category")

    max_date    = model_df["date"].max()
    valid_start = max_date - pd.Timedelta(days=VALID_DAYS - 1)
    train_df = model_df[model_df["date"] < valid_start].copy()
    valid_df  = model_df[model_df["date"] >= valid_start].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["demand"]
    X_valid = valid_df[feature_cols]

    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
    lgb_valid = lgb.Dataset(X_valid, label=valid_df["demand"], categorical_feature=cat_cols, free_raw_data=False)

    model = lgb.train(
        LGB_PARAMS,
        lgb_train,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        num_boost_round=NUM_BOOST_ROUND,
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
            lgb.log_evaluation(period=VERBOSE_EVAL)
        ]
    )

    valid_df["pred"] = model.predict(X_valid, num_iteration=model.best_iteration)
    rmse = root_mean_squared_error(valid_df["demand"], valid_df["pred"])
    mae  = mean_absolute_error(valid_df["demand"], valid_df["pred"])

    row = {
        "exp": exp_name,
        "added_features": ", ".join(experiment_addons[exp_name]),
        "rmse": rmse,
        "mae": mae,
        "n_rows": model_df.shape[0],
        "n_features": len(feature_cols)
    }
    results.append(row)

    # Save per-experiment files
    pd.DataFrame([row]).to_csv(OUTPUT_DIR / f"{PIPELINE_NAME}_{exp_name}_metrics.csv", index=False)
    valid_df[["id", "date", "demand", "pred"]].to_csv(OUTPUT_DIR / f"{PIPELINE_NAME}_{exp_name}_valid_predictions.csv", index=False)
    pd.DataFrame({
        "feature": feature_cols,
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split")
    }).sort_values("importance_gain", ascending=False).to_csv(
        OUTPUT_DIR / f"{PIPELINE_NAME}_{exp_name}_feature_importance.csv", index=False
    )

    print(f"RMSE: {rmse:.6f} | MAE: {mae:.6f} | best_iter: {model.best_iteration}")
    gc.collect()

print("\n" + "=" * 70)
print("ALL EXPERIMENTS DONE")
results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(results_df)

Running: E5 | Added: ['price_vs_mean', 'price_vs_dept_avg']
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 1.98123	valid's rmse: 2.16061
Early stopping, best iteration is:
[99]	train's rmse: 1.98247	valid's rmse: 2.15899
RMSE: 2.158988 | MAE: 1.017276 | best_iter: 99
Running: E6 | Added: ['is_price_drop', 'is_price_min_7d']
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 1.98192	valid's rmse: 2.16078
[200]	train's rmse: 1.93737	valid's rmse: 2.17167
Early stopping, best iteration is:
[102]	train's rmse: 1.97983	valid's rmse: 2.16047
RMSE: 2.160473 | MAE: 1.016488 | best_iter: 102
Running: E7 | Added: ['snap_flag', 'is_event']
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 1.96603	valid's rmse: 2.14511
[200]	train's rmse: 1.91118	valid's rmse: 2.14844
Early stopping, best iteration is:
[122]	train's rmse: 1.94537	valid's rmse: 2.14212
RMSE: 2.142124 | MAE: 1.012065 | best_iter: 122
Ru

,exp,added_features,rmse,mae,n_rows,n_features
0,E9,"price_vs_mean, price_vs_dept_avg, is_price_dro...",2.136403,1.012673,946000,22
1,E7,"snap_flag, is_event",2.142124,1.012065,946000,16
2,E5,"price_vs_mean, price_vs_dept_avg",2.158988,1.017276,946000,16
3,E8,"dept_avg_demand, store_avg_demand",2.159977,1.017729,946000,16
4,E6,"is_price_drop, is_price_min_7d",2.160473,1.016488,946000,16


In [10]:
# =========================
# 10) Save combined summary
# =========================
summary_path = OUTPUT_DIR / f"{PIPELINE_NAME}_E5_E9_summary.csv"
results_df.to_csv(summary_path, index=False)
print("Summary saved to:", summary_path)
display(results_df)

Summary saved to: /content/drive/MyDrive/Group Project - Predictive/data/FE_output/baseline_lgb_clean_minimal_features_E5_E9_summary.csv


,exp,added_features,rmse,mae,n_rows,n_features
0,E9,"price_vs_mean, price_vs_dept_avg, is_price_dro...",2.136403,1.012673,946000,22
1,E7,"snap_flag, is_event",2.142124,1.012065,946000,16
2,E5,"price_vs_mean, price_vs_dept_avg",2.158988,1.017276,946000,16
3,E8,"dept_avg_demand, store_avg_demand",2.159977,1.017729,946000,16
4,E6,"is_price_drop, is_price_min_7d",2.160473,1.016488,946000,16
